# Эволюция машинок

Здесь мы научим целую «стаю» машинок проходить круг по трассе, ни разу не включив обучение с учителем и не задав ни одного правила вождения. Вместо этого используем идею естественного отбора.

**Один цикл (поколение):**

1. **Оценить.** Каждая машинка из популяции едет по трассе со своим «мозгом» — небольшой нейросетью. В конце заезда у каждой есть число — *фитнес* (насколько хорошо она проехала).
2. **Отобрать.** Лучшие машинки (*элита*) переходят в следующее поколение без изменений.
3. **Мутировать.** Остальные места занимают слегка изменённые копии лучших: к части весов сети добавляется небольшой случайный шум. Часть потомков окажется чуть лучше родителей, и со временем популяция учится ехать дальше.

Затем всё повторяется с новой популяцией. Кривая «лучший прогресс по поколениям» должна ползти вверх.

**Что делаешь ты, а что даёт библиотека.**

- Библиотека (`rl_fun.racing`) даёт трассу, физику, лучи-датчики, много машинок сразу (`RacingFleet`), прогон одного поколения (`run_generation`) и окно с картинкой (`FleetView`).
- Ты пишешь четыре функции: `init_population` (стартовые веса), `forward` (как сеть превращает наблюдение в действие), `next_generation` (отбор и мутация) и, по желанию, `inspect` (чтобы увидеть сеть в окне). В ноутбуке они помечены «Заготовка».
- В конце можно сравнить свою кривую с эталоном, но не подглядывай раньше времени: это интереснее, когда пытаешься сам.

Запускай ячейки по порядку сверху вниз.

## Параметры

Все настройки собраны в одной ячейке. Меняй значения и перезапускай обучение.

- `TRACK` — трасса: `"oval"` (простой овал) или `"wavy"` (волнистая, сложнее).
- `RAY_ANGLES` — углы лучей-датчиков в градусах относительно направления машинки: `0` — прямо вперёд, `±90` — строго вбок. Чем больше лучей, тем лучше машинка «видит», но тем больше весов в сети.
- `POPULATION` — сколько машинок едет одновременно. Больше машинок — больше разнообразия, но дольше считается.
- `GENERATIONS` — сколько поколений обучаем.
- `MAX_STEPS` — предел длины заезда в шагах симуляции (30 шагов = 1 секунда). Заезд кончается раньше, если все машинки разбились или финишировали.
- `HIDDEN` — размеры скрытых слоёв нейросети, например `(6, 5)` — два слоя по 6 и 5 нейронов.
- `SEED` — зерно генератора случайных чисел. С тем же зерном получится тот же результат.
- `SHOW_WINDOW` — показывать ли окно с машинками. Окно сильно замедляет обучение; поставь `False`, чтобы считать быстро.
- `SHOW_NETWORK` — рисовать ли рядом с трассой панель с нейросетью лидера. Работает, только если ты реализовал `inspect` (Заготовка 4).

In [ ]:
TRACK = "oval"
RAY_ANGLES = (-90, -30, 0, 30, 90)
POPULATION = 50
GENERATIONS = 60
MAX_STEPS = 1500
HIDDEN = (6, 5)
SEED = 7
SHOW_WINDOW = True
SHOW_NETWORK = False

## Флот и наблюдения

`RacingFleet` — это много машинок на одной трассе, которые считаются разом. Сначала разберёмся, что видит каждая машинка и что она умеет.

**Наблюдение** — вектор чисел, описывающий текущую ситуацию. Он состоит из двух частей:

- **Лучи.** Из машинки выходят лучи под углами `RAY_ANGLES`. Каждый луч измеряет расстояние до края дороги. Расстояние делится на максимальную дальность луча, поэтому число лежит в $[0, 1]$: `0` — край прямо перед носом, `1` — свободно как минимум до предела видимости.
- **Три скорости.** Продольная скорость (вперёд) в долях максимальной, лежит в $[0, 1]$. Боковая скорость (снос) и угловая скорость поворота нормированы так, что лежат в $[-1, 1]$.

Итого $\text{размер наблюдения} = \text{число лучей} + 3$. Нормировка важна: когда все входы одного порядка, сети проще подбирать веса.

**Действие** — два числа в диапазоне $[-1, 1]$: `[руль, газ]`. Руль `-1` и `+1` — предельный поворот в разные стороны, `0` — прямо. Газ `+1` — полный газ, отрицательный газ замедляет машинку. Значения вне диапазона библиотека обрезает.

**Фитнес** — число, по которому мы выбираем лучших. По умолчанию это `progress`: сколько кругов машинка проехала по трассе вдоль её центральной линии (`1.0` — полный круг). Почему не «сколько живёт» или «какая скорость»? Прогресс прямо измеряет то, что нам нужно, и его сложно «накрутить»: машинка, которая стоит на месте или кружит на одном месте, прогресса не получит. Позже ты сможешь придумать свою форму фитнеса.

Машинка «умирает» (замирает) при выезде за границу дороги и тоже замирает, когда проехала круг.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from rl_fun.racing.fleet import RacingFleet
from rl_fun.racing.fleet_view import FleetView, ViewClosed
from rl_fun.racing.generation import run_generation

fleet = RacingFleet(POPULATION, track=TRACK, ray_angles_deg=RAY_ANGLES, max_steps=MAX_STEPS)
observation = fleet.reset()

print("Форма наблюдений (машинок, чисел на машинку):", observation.shape)
print("Наблюдение первой машинки:", np.round(observation[0], 2))
print("Размер наблюдения:", fleet.observation_size)

Теперь посчитаем, из каких слоёв состоит сеть и сколько в ней весов.

Сеть — это цепочка слоёв. Размеры слоёв: вход (размер наблюдения), скрытые слои из `HIDDEN`, выход из двух чисел (руль и газ). Между соседними слоями размером $a$ и $b$ есть $a \times b$ весов связей, а у каждого нейрона слоя $b$ есть ещё одно число — **смещение** (bias). Поэтому для пары слоёв получается $(a + 1) \cdot b$ параметров, а всего:

$$N_{\text{weights}} = \sum_{i} (a_i + 1)\, b_i.$$

Все параметры одной сети мы храним в **одном плоском векторе** длины `N_WEIGHTS`. Вся популяция — это массив формы `(POPULATION, N_WEIGHTS)`: строка на машинку. Ровно такой формат ждёт `run_generation`.

In [ ]:
SIZES = [fleet.observation_size, *HIDDEN, 2]
N_WEIGHTS = sum((a + 1) * b for a, b in zip(SIZES[:-1], SIZES[1:], strict=True))

print("Размеры слоёв:", SIZES)
print("Весов и смещений в одной сети:", N_WEIGHTS)

## Заготовка 1: начальная популяция

Заполни функцию `init_population`: она создаёт стартовые веса для всей популяции.

**Контракт:**

- `rng` — генератор случайных чисел `np.random.Generator` (его создаём в цикле обучения от `SEED`; бери случайность только из него, чтобы результат был воспроизводимым).
- `n` — размер популяции, `n_weights` — число весов одной сети.
- Верни `np.ndarray` формы `(n, n_weights)`: строка `i` — веса $i$-й машинки.

**Про масштаб.** Веса должны быть случайными и разными у разных машинок (иначе все поедут одинаково и отбирать будет не из чего), но не огромными: слишком большие числа сразу «насыщают» сеть, и она всегда выдаёт крайние значения. Начни с чисел порядка единицы или меньше и поэкспериментируй.

In [ ]:
def init_population(rng, n, n_weights):
    # Верни массив формы (n, n_weights) со случайными небольшими весами.
    raise NotImplementedError("впиши здесь создание популяции: массив формы (n, n_weights)")

## Заготовка 2: сеть

Заполни функцию `forward`: она превращает наблюдение в действие, используя веса одной машинки.

**Контракт:**

- `weights` — плоский вектор длины `N_WEIGHTS` (одна строка популяции).
- `observation` — вектор длины `SIZES[0]` (см. «Флот и наблюдения»).
- Верни вектор из двух чисел `[руль, газ]`, каждое строго в диапазоне $[-1, 1]$.

**Подсказки.**

- Сеть — это цепочка слоёв $a^{(k+1)} = f\big(W^{(k)} a^{(k)} + b^{(k)}\big)$, где $W^{(k)}$ — матрица весов, $b^{(k)}$ — смещения, $f$ — функция активации. Размеры слоёв лежат в `SIZES`.
- Тебе нужно самому решить, как «нарезать» плоский вектор `weights` на матрицы и смещения. Удобно идти по слоям и отрезать нужное число элементов. Дальше (в `inspect`) пригодится, чтобы матрица каждого слоя имела форму `(выходов, входов)`.
- Функция активации нужна не только как «нелинейность»: выход сети должен попадать в $[-1, 1]$. Подумай, какие функции ограничены этим диапазоном.
- `forward` вызывается для каждой живой машинки на каждом шаге, поэтому она должна быть быстрой: без циклов по нейронам.

In [ ]:
def forward(weights, observation):
    # Верни действие [руль, газ] со значениями в [-1, 1].
    raise NotImplementedError("впиши здесь прямой проход: наблюдение -> [руль, газ] в [-1, 1]")

## Заготовка 3: отбор и мутация

Заполни функцию `next_generation`: она строит следующее поколение по текущему.

**Контракт:**

- `population` — массив `(N, W)`, строка на машинку.
- `fitness` — массив `(N,)`, `fitness[i]` — оценка $i$-й машинки; **чем больше, тем лучше**.
- `rng` — генератор случайных чисел (`np.random.Generator`).
- Верни **новый** массив той же формы `(N, W)`. Размер популяции не должен меняться. Входной `population` лучше не менять на месте.

**Идея** (подробнее — во введении): лучшие переходят дальше без изменений (*элита*), остальные места занимают мутировавшие копии лучших.

**Что нужно решить самому:**

- Сколько машинок в элите. Если элита слишком большая, новому не из чего вырасти; слишком маленькая — легко потерять хорошее решение.
- Какие веса мутировать: все или только случайную долю.
- Насколько сильно мутировать. Слишком слабая мутация — обучение ползёт; слишком сильная — потомки не похожи на родителей, и польза от отбора пропадает.
- Откуда брать родителей для потомков: например, только из элиты.

Проверь себя: веса лучшей машинки не должны пропасть при смене поколения.

In [ ]:
def next_generation(population, fitness, rng):
    # Верни новый массив той же формы (N, W): элита без изменений + мутировавшие потомки.
    raise NotImplementedError("впиши здесь отбор и мутацию: верни массив формы (N, W)")

## Заготовка 4 (необязательно): взгляд внутрь сети

Эта функция нужна только для красивой панели в окне: она показывает нейросеть лидера, её веса и то, как «загораются» нейроны. Без неё обучение работает точно так же.

**Контракт:** `inspect(weights, observation)` возвращает пару `(matrices, activations)`:

- `matrices` — список матриц весов по слоям, **без смещений**. Матрица слоя имеет форму `(число выходов слоя, число входов слоя)`. Всего `len(SIZES) - 1` матриц.
- `activations` — список векторов значений нейронов по слоям: сначала сам вход (наблюдение), потом выход каждого слоя до самого последнего (это и есть `[руль, газ]`). Всего `len(SIZES)` векторов.

То есть это тот же прямой проход, что и в `forward`, только с запоминанием промежуточных результатов.

**Как увидеть панель:** реализуй функцию, поставь `SHOW_NETWORK = True` (и `SHOW_WINDOW = True`) в ячейке параметров и перезапусти обучение. Зелёные связи — положительные веса, красные — отрицательные, чем толще связь, тем больше её вес по модулю. Цвет узла показывает знак активации.

In [ ]:
def inspect(weights, observation):
    # Верни (matrices, activations) как описано выше.
    raise NotImplementedError("впиши здесь прямой проход: верни (matrices, activations)")

## Обучение

Теперь соберём всё вместе. Цикл поколений делает следующее:

1. `run_generation` прогоняет популяцию по трассе (вызывает твою `forward` для каждой живой машинки на каждом шаге) и возвращает результат: `progress`, `finished` (кто проехал круг), `steps_alive` и другое.
2. Из результата берём **фитнес**. По умолчанию это прогресс (в кругах). В строке с комментарием «можно изменить» ты позже можешь подставить свою формулу: например, добавить бонус за скорость прохождения круга.
3. Записываем в `history` лучший и средний прогресс и число финишировавших.
4. `next_generation` строит новую популяцию.

Если закрыть окно, обучение остановится, но уже накопленные `history` и лучшие веса останутся для следующих ячеек. Во время обучения в окне видно, сколько машинок ещё живо; лидер выделен, лучи рисуются только у него.

In [ ]:
rng = np.random.default_rng(SEED)
fleet = RacingFleet(POPULATION, track=TRACK, ray_angles_deg=RAY_ANGLES, max_steps=MAX_STEPS)
population = init_population(rng, POPULATION, N_WEIGHTS)
assert population.shape == (POPULATION, N_WEIGHTS), (
    f"init_population должна вернуть форму {(POPULATION, N_WEIGHTS)}, а вернула {population.shape}"
)

view = FleetView(fleet, show_network=SHOW_NETWORK) if SHOW_WINDOW else None

history = []
best_weights = population[0].copy()
best_fitness = -np.inf
try:
    for generation in range(GENERATIONS):
        result = run_generation(
            fleet,
            population,
            forward,
            view=view,
            inspect=inspect if SHOW_NETWORK else None,
            label=f"Поколение {generation + 1}/{GENERATIONS}",
        )
        fitness = result.progress  # можно изменить

        history.append(
            {
                "generation": generation,
                "best": float(result.progress.max()),
                "mean": float(result.progress.mean()),
                "finished": int(result.finished.sum()),
            }
        )
        print(
            f"Поколение {generation + 1:3d}: лучший {history[-1]['best']:.3f}, "
            f"средний {history[-1]['mean']:.3f}, финишировало {history[-1]['finished']}"
        )

        top = int(np.argmax(fitness))
        if fitness[top] > best_fitness:
            best_fitness = float(fitness[top])
            best_weights = population[top].copy()

        population = next_generation(population, fitness, rng)
        assert population.shape == (POPULATION, N_WEIGHTS), (
            f"next_generation должна вернуть форму {(POPULATION, N_WEIGHTS)}, "
            f"а вернула {population.shape}"
        )
except ViewClosed:
    print("Окно закрыто, обучение остановлено.")
finally:
    if view is not None:
        view.close()

## Кривая обучения

Как читать график:

- **Лучший прогресс** показывает, как далеко уехала самая успешная машинка поколения. Значение около `1.0` — полный круг.
- **Средний прогресс** показывает, насколько хороша вся популяция, а не только лучшая. Если лучший растёт, а средний стоит на месте, элита уходит вперёд, а остальные отстают: возможно, мутация слишком сильная.
- График может ступенчато расти и временно стоять на месте: это нормально, эволюция идёт рывками, когда находит новое «умение» (например, поворот).
- Если кривая вообще не растёт, проверь масштаб начальных весов, силу мутации и размер элиты.

In [ ]:
generations = [entry["generation"] + 1 for entry in history]
plt.figure(figsize=(8, 4.5))
plt.plot(generations, [entry["best"] for entry in history], label="лучший")
plt.plot(generations, [entry["mean"] for entry in history], "--", label="средний")
plt.xlabel("Поколение")
plt.ylabel("Прогресс (кругов)")
plt.title("Кривая обучения")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Сравнение с эталоном

В библиотеке есть эталонное решение: небольшая сеть и простая эволюция, обученные на этой же трассе с зерном `7`. Здесь мы сравним твою кривую с его сохранённой кривой.

Это не соревнование на «победу», а ориентир: если ты заметно отстаёшь, посмотри, чем отличаются размер элиты, мутация и масштаб начальных весов.

Эталон есть не для каждой трассы. Если для твоей `TRACK` его нет, ячейка сообщит об этом ошибкой.

In [ ]:
from rl_fun.racing.benchmark import compare, load_benchmark, plot_comparison

benchmark = load_benchmark(TRACK)
print(compare(history, benchmark))
plot_comparison(history, benchmark)

## Посмотреть лучшую машинку и эталон

Посмотрим на лучшую найденную тобой машинку: она едет одна по пустой трассе с теми весами, которые получили наибольший фитнес за всё обучение. Ячейка работает, только если `SHOW_WINDOW = True`.

In [ ]:
if SHOW_WINDOW:
    solo = RacingFleet(1, track=TRACK, ray_angles_deg=RAY_ANGLES, max_steps=MAX_STEPS)
    solo_view = FleetView(solo, show_network=SHOW_NETWORK)
    try:
        run_generation(
            solo,
            best_weights[np.newaxis, :],
            forward,
            view=solo_view,
            inspect=inspect if SHOW_NETWORK else None,
            label="Твоя лучшая машинка",
        )
    except ViewClosed:
        print("Окно закрыто.")
    finally:
        solo_view.close()

А теперь для сравнения запустим эталонную машинку с сохранёнными весами. Она едет с теми же настройками трассы, что и при создании эталона.

In [ ]:
if SHOW_WINDOW:
    from rl_fun.racing import reference

    ref_fleet = RacingFleet(1, track=TRACK, **benchmark["fleet"])
    ref_view = FleetView(ref_fleet, show_network=True)
    ref_sizes = benchmark["sizes"]
    ref_weights = np.asarray(benchmark["best_weights"])
    try:
        run_generation(
            ref_fleet,
            ref_weights[np.newaxis, :],
            lambda w, o: reference.forward(w, o, ref_sizes),
            view=ref_view,
            inspect=lambda w, o: reference.inspect(w, o, ref_sizes),
            label="Эталонная машинка",
        )
    except ViewClosed:
        print("Окно закрыто.")
    finally:
        ref_view.close()

## Что попробовать дальше

- **Другое число лучей.** Измени `RAY_ANGLES`, например добавь лучи `±60` или оставь только три. Не забудь, что вместе с ним поменяется размер входа и число весов.
- **Архитектура сети.** Поиграй с `HIDDEN`: один слой побольше, три маленьких, вообще без скрытых слоёв. Что лучше при том же числе поколений?
- **Форма фитнеса.** Замени строку `fitness = result.progress  # можно изменить`. Например, добавь бонус тем, кто проехал круг быстрее (подсказка: массив `result.lap_steps`, у не финишировавших в нём `nan`), либо штраф за долгое стояние на месте.
- **Трасса `wavy`.** Попробуй `TRACK = "wavy"` и посмотри, насколько хватает переноса умений. Сравнение с эталоном для неё может быть недоступно.
- **Скорость мутации.** Как меняется кривая, если мутировать сильнее или реже? Есть ли «золотая середина»?
- **Размер популяции и элиты.** Больше машинок или меньше, больше элиты или меньше, какое сочетание работает лучше?
- **Разные зёрна.** Прогони обучение с несколькими `SEED` и сравни кривые: насколько результат зависит от везения?